## Background
This notebook provides the solution for span identification task.

## Imports

In [ ]:
import warnings
import ast
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import Counter
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score
from sklearn.model_selection import StratifiedKFold
from transformers import RobertaTokenizerFast, AutoModel, Trainer, TrainingArguments, EvalPrediction
from torch.utils.data import Dataset, DataLoader, random_split
import torch
import torch.nn as nn
import torch.nn.functional as F
from safetensors.torch import load_file as safe_load_file

In [ ]:
warnings.filterwarnings("ignore")

## Constants

In [1]:
TRAIN_PATH = "../../data/"
TRAIN_NAME = "train.parquet"
TEST_NAME = "test.csv"

MODEL_NAME = "FacebookAI/xlm-roberta-large"

In [ ]:
if torch.cuda.is_available():
    DEVICE = torch.DEVICE("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print(f"Using device: {DEVICE}")

Using device: cuda


## Read data

In [9]:
train = pd.read_parquet(os.path.join(TRAIN_PATH, TRAIN_NAME))
display(train.head())

test = pd.read_csv(os.path.join(TRAIN_PATH, TEST_NAME))
display(test.head())

,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


,id,content,techniques,trigger_words
0,521cd2e8-dd9f-42c4-98ba-c0c8890ff1ba,"Они просрали нашу технику, положили кучу людей...","['fud', 'loaded_language']","[(0, 12), (27, 46), (48, 71), (131, 162), (164..."
1,9b2a61e4-d14e-4ff7-b304-e73d720319bf,❗️\nКитай предлагает отдать оккупированные тер...,['loaded_language'],"[(374, 425)]"
2,f0f1c236-80a8-4d25-b30c-a420a39be632,Сегодня будет ровно 6 месяцев с этого обещания...,['loaded_language'],"[(0, 127)]"
3,31ea05ba-2c2b-4b84-aba7-f3cf6841b204,⚡️\nІзраїль вперше у світі збив балістичну рак...,[],NaN
4,a79e13ec-6d9a-40b5-b54c-7f4f743a7525,Склав невелику навчально-методичну таблицю на ...,['loaded_language'],"[(87, 103), (127, 136), (170, 189), (204, 255)..."


In [15]:
len(test)

5735

In [4]:
test['trigger_words'] = test['trigger_words'].apply(lambda x: x if isinstance(x, str) else "[]")
test['trigger_words'] = test['trigger_words'].apply(ast.literal_eval)
test['techniques'] = test['techniques'].apply(ast.literal_eval)

In [5]:
from collections import Counter

unique_techniques = Counter()
for t in train["techniques"]:
    unique_techniques.update(t)

unique_techniques

Counter({'loaded_language': 1973,
         'cherry_picking': 512,
         'glittering_generalities': 483,
         'cliche': 463,
         'euphoria': 462,
         'fud': 385,
         'appeal_to_fear': 300,
         'whataboutism': 158,
         'bandwagon': 157,
         'straw_man': 138})

In [6]:
# Extract trigger_words phrases from the text:
trigger_words_phrases = []
for text, trigger_words in zip(train.content.values, train.trigger_words.values):
    if text is None or trigger_words is None:
        trigger_words_phrases.append([])
        continue
    trigger_words_phrases.append([text[i:j] for i, j in trigger_words])
train["trigger_words_phrases"] = trigger_words_phrases

## Dataset

In [ ]:
tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, use_fast=True)

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLMRobertaTokenizerFast'. 
The class this function is called from is 'RobertaTokenizerFast'.


In [8]:
class ManipulationDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=512):
        self.data = df
        self.tokenizer = tokenizer
        self.max_length = max_length

        # Extract unique labels
        all_labels = set()
        self.data['techniques'].apply(lambda x: all_labels.update(x))
        self.label2id = {label: idx for idx, label in enumerate(sorted(all_labels))}
        self.id2label = {idx: label for label, idx in self.label2id.items()}

        self.num_labels = len(all_labels)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        text = self.data.iloc[idx]['content']
        labels = self.data.iloc[idx]['techniques']  # Multi-label targets
        spans = self.data.iloc[idx]['trigger_words']  # Character-based spans

        # Tokenize input text
        encoding = self.tokenizer(text, padding='max_length', truncation=True, max_length=self.max_length, return_tensors="pt", return_offsets_mapping=True)
        input_ids = encoding['input_ids'].squeeze()
        attention_mask = encoding['attention_mask'].squeeze()
        offset_mapping = encoding['offset_mapping'].squeeze(0)   # shape: (max_length, 2)

        # Convert multi-label to tensor
        class_labels = torch.zeros(self.num_labels, dtype=torch.float)
        for label in labels:
            if label in self.label2id:
                class_labels[self.label2id[label]] = 1

        # Initialize token-level labels.
        # We'll mark tokens that are not real (i.e. padded tokens) as -100.
        token_labels = torch.full((self.max_length,), -100, dtype=torch.long)
        # For tokens that are not padding, set default label 0 (non-manipulative).
        for i in range(self.max_length):
            if attention_mask[i] == 1:
                token_labels[i] = 0

        # Loop over each token using its offset mapping.
        # If the token (defined by its character span) overlaps with any trigger span, label it as 1.
        # print(self.data.iloc[idx]['trigger_words_phrases'])
        for i, (token_start, token_end) in enumerate(offset_mapping.tolist()):
            # Skip pad tokens
            if attention_mask[i] == 0:
                continue
            for span in spans:
                span_start, span_end = span
                # Check if there is any overlap between token span and trigger span.
                if token_end > span_start and token_start < span_end:
                    # print(text[token_start:token_end])
                    token_labels[i] = 1
                    break  # No need to check other spans for this token.


        return {"input_ids": input_ids, "attention_mask": attention_mask, "class_labels": class_labels, "token_labels": token_labels}

In [ ]:
train['techniques'] = train['techniques'].apply(lambda x: [] if x is None else x)
train['trigger_words'] = train['trigger_words'].apply(lambda x: [] if x is None else x)

In [14]:
full_dataset = ManipulationDataset(train, tokenizer)
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size
train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size], generator=torch.Generator().manual_seed(42))

## Model

In [ ]:
import wandb
wandb.init(mode="disabled")

In [21]:
class SequenceClassificationHead(nn.Module):
    def __init__(self, hidden_size, num_labels, dropout_prob=0.1, activation_fn="gelu"):
        super().__init__()

        layer_size = 256

        self.dense = nn.Linear(hidden_size, layer_size)
        self.activation = nn.GELU()

        self.layer_norm = nn.LayerNorm(layer_size)
        self.dropout = nn.Dropout(dropout_prob)
        self.output_layer = nn.Linear(layer_size, num_labels)

    def forward(self, x):
        # Apply dense layer and activation
        x = self.dense(x)
        x = self.activation(x)

        # Apply layer normalization and dropout
        x = self.layer_norm(x)
        x = self.dropout(x)

        # Output layer
        x = self.output_layer(x)
        return x

In [ ]:
class MultiTaskModel(nn.Module):
    def __init__(
        self,
        model_name="xlm-roberta-base",
        num_labels=10,
        class_loss_weight=1.0,
        token_loss_weight=1.0,
        dropout_prob=0.1,
    ):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name, add_pooling_layer=False)
        hidden_size = self.encoder.config.hidden_size

        # Multi-label classification head
        self.classification_head = SequenceClassificationHead(hidden_size * 3, num_labels, dropout_prob=dropout_prob)

        # Token classification head (binary classification with single output)
        self.token_classification_head = nn.Linear(hidden_size, 1)  # Single output for binary classification

        # Loss weights
        self.class_loss_weight = class_loss_weight
        self.token_loss_weight = token_loss_weight

    def forward(self, input_ids, attention_mask, class_labels=None, token_labels=None):
        # Get encoder outputs
        outputs = self.encoder(input_ids, attention_mask=attention_mask)
        sequence_output = outputs.last_hidden_state  # (batch_size, seq_len, hidden_size)

        # Combine pooling strategies: [CLS], mean, and max pooling
        cls_pooled = sequence_output[:, 0]  # [CLS] token (batch_size, hidden_size)
        mean_pooled = sequence_output.mean(dim=1)  # Mean pooling (batch_size, hidden_size)
        max_pooled = sequence_output.max(dim=1).values  # Max pooling (batch_size, hidden_size)

        # Concatenate pooled outputs
        combined_pooled = torch.cat([cls_pooled, mean_pooled, max_pooled], dim=-1)  # (batch_size, hidden_size * 3)

        # Compute logits
        class_logits = self.classification_head(combined_pooled)  # Multi-label classification logits
        token_logits = self.token_classification_head(sequence_output)  # Token classification logits (batch_size, seq_len, 1)

        # Calculate losses if labels are provided
        loss = None
        if class_labels is not None and token_labels is not None:
            # Multi-label classification loss (BCEWithLogitsLoss for multi-label tasks)
            class_loss = F.binary_cross_entropy_with_logits(class_logits, class_labels)

            # Token classification loss (BCEWithLogitsLoss for binary classification)
            token_loss_fn = nn.BCEWithLogitsLoss()
            # Flatten predictions and labels
            flat_token_logits = token_logits.view(-1)
            flat_token_labels = token_labels.view(-1)

            # Create a mask for valid (non-pad) tokens
            active_mask = flat_token_labels != -100
            active_logits = flat_token_logits[active_mask]
            active_labels = flat_token_labels[active_mask].float()
            token_loss = token_loss_fn(active_logits, active_labels)

            # Weighted sum of both losses
            loss = self.class_loss_weight * class_loss + self.token_loss_weight * token_loss

        return {
            "loss": loss,
            "class_logits": class_logits,
            "token_logits": token_logits,
        }

In [25]:
batch_size = 16

In [ ]:
def get_f1_macro_thr_cv_stratified_multilabel(targets, outputs, n_splits=5):
    """
    Computes the best threshold for each class using stratified K-fold cross-validation.
    """
    num_targets = targets.shape[1]
    avg_thresholds = np.zeros(num_targets)

    # Ignore warnings (useful for cases where f1_score might issue warnings)
    warnings.filterwarnings("ignore")

    for i in range(num_targets):
        skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
        thresholds = []

        for train_idx, val_idx in skf.split(np.zeros(len(targets)), targets[:, i]):
            val_targets, val_outputs = targets[val_idx, i], outputs[val_idx, i]
            best_f1, best_thresh = 0, 0.01

            for threshold in np.arange(0.01, 0.99, 0.01):
                preds = (val_outputs > threshold).astype(int)
                f1 = f1_score(val_targets, preds)
                if f1 > best_f1:
                    best_f1, best_thresh = f1, threshold

            thresholds.append(best_thresh)
        # print("All thresholds: ", thresholds)
        avg_thresholds[i] = np.median(thresholds)  # Average thresholds across folds

    # Compute final validation predictions using averaged thresholds
    validation_predictions = (outputs > avg_thresholds).astype(int)
    final_f1 = f1_score(targets, validation_predictions, average='macro')

    print("Stable averaged thresholds across folds:", avg_thresholds)
    print("Final F1-macro:", final_f1)

    return avg_thresholds, final_f1


def get_f1_macro_thr_cv_stratified_token(targets, outputs, n_splits=5):
    """
    Compute the best threshold for binary classification using stratified K-fold cross-validation.
    """
    # Ignore warnings (useful for cases where f1_score might issue warnings)
    warnings.filterwarnings("ignore")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    thresholds = []
    metrics = []

    for train_idx, val_idx in skf.split(targets, targets):
        val_targets, val_outputs = targets[val_idx], outputs[val_idx]
        val_targets_flat = val_targets.flatten()
        val_outputs_flat = val_outputs.flatten()

        best_f1, best_thresh = 0, 0.01
        for threshold in np.arange(0.01, 0.99, 0.01):
            preds = (val_outputs > threshold).astype(int)
            f1 = f1_score(val_targets, preds)
            if f1 > best_f1:
                best_f1, best_thresh = f1, threshold

        thresholds.append(best_thresh)
        metrics.append(best_f1)

    avg_threshold = np.median(thresholds)  # Average thresholds across folds
    # Compute final validation predictions using averaged thresholds
    targets_flat = targets.flatten()
    outputs_flat = outputs.flatten()


    validation_predictions = (outputs_flat > avg_threshold).astype(int)
    final_f1 = f1_score(targets_flat, validation_predictions)

    print("Stable averaged threshold across folds:", avg_threshold)
    print("Fold metrics: ", metrics)
    print("Final F1-macro:", final_f1)
    return avg_threshold, final_f1



def compute_metrics(eval_pred):

    logits, labels = eval_pred  # Trainer returns a tuple with logits and labels

    class_logits, token_logits = logits
    class_labels, token_labels = labels

    # class_preds = (class_logits > 0).astype(int)  # Convert logits to binary predictions
    outputs = torch.sigmoid(torch.tensor(class_logits)).cpu().numpy()

    # Compute F1 score for multi-label classification (macro-average)
    _, class_f1 = get_f1_macro_thr_cv_stratified_multilabel(class_labels, outputs)

    # Compute F1 score for token classification (only for non-padding tokens)
    valid_indices = token_labels != -100  # Ignore padding tokens
    token_preds = torch.sigmoid(torch.tensor(token_logits)).cpu().numpy()
    _, token_f1 = get_f1_macro_thr_cv_stratified_token(token_labels[valid_indices], token_preds[valid_indices])

    overall_f1 = (class_f1 + token_f1) / 2

    return {
        "class_f1": class_f1,
        "token_f1": token_f1,
        "overall_f1": overall_f1
    }

In [ ]:
metric_name = "class_f1"

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    # evaluation_strategy = "steps",
    evaluation_strategy = "epoch",
    # eval_steps = 100,
    save_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=10,
    weight_decay=0.01,
    # load_best_model_at_end=True,
    metric_for_best_model=metric_name,
    greater_is_better=True,
)

/venv/main/lib/python3.10/site-packages/transformers/training_args.py:1594: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [30]:
trainer = Trainer(
    model=MultiTaskModel(
        num_labels=10,
        model_name=MODEL_NAME,
        class_loss_weight=0.7,
        token_loss_weight=0.3,
        dropout_prob=0.1,
    ),
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

/tmp/ipykernel_656/400109280.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
trainer.train()

In [ ]:
trainer.evaluate()

# Inference logic

In [ ]:
# thresholds are chosen manually are review of the training results

MODEL_DUMP = "/content/results/checkpoint-960"
model = MultiTaskModel(
        num_labels=10,
        model_name=MODEL_NAME,
        class_loss_weight=0.7,
        token_loss_weight=0.3,
        dropout_prob=0.1,
    )
# checkpoint_path = "checkpoint-768/model.safetensors"
state_dict = safe_load_file(MODEL_DUMP + "/model.safetensors")
model.load_state_dict(state_dict)
model.to(DEVICE)

tokenizer = RobertaTokenizerFast.from_pretrained(MODEL_NAME, use_fast=True)

# class_thresholds = np.array([0.25, 0.17 ,0.21, 0.18 ,0.28, 0.52 ,0.28, 0.26 ,0.2 , 0.16])
# token_threshold = 0.13

The tokenizer class you load from this checkpoint is not the same type as the class this function is called from. It may result in unexpected tokenization. 
The tokenizer class you load from this checkpoint is 'XLMRobertaTokenizer'. 
The class this function is called from is 'RobertaTokenizerFast'.


In [ ]:
def run_inference(model, tokenizer, text, max_length=512, class_thresholds=[], token_threshold=0.5, device=DEVICE):

    # Determine device if not provided
    if device is None:
        device = next(model.parameters()).device

    # Set model to evaluation mode
    model.eval()

    # Tokenize input text
    encoding = tokenizer(
        text,
        padding='max_length',
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        return_offsets_mapping=True
    )
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    offset_mapping = encoding['offset_mapping'][0].tolist()

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)

    # Process classification head outputs
    class_logits = outputs["class_logits"]  # (batch_size, num_labels)
    class_probs = torch.sigmoid(torch.tensor(class_logits)).cpu().numpy()
    class_preds = (class_probs > class_thresholds).astype(int)


    # Process token classification head.
    # token_logits shape: (batch_size, seq_length, 1)
    token_logits = outputs["token_logits"].squeeze(-1)  # Now shape: (batch_size, seq_length)
    token_probs = torch.sigmoid(torch.tensor(token_logits)).cpu().numpy()
    token_preds = (token_probs > token_threshold)
    # Mask out padded tokens.
    token_preds = token_preds * attention_mask.cpu().numpy()

    # For span reconstruction, assume batch size 1.
    token_preds = token_preds[0].tolist()  # List of predictions per token.

    # Convert token predictions into spans using the offset mapping.
    spans = []
    current_span = None
    for pred, (tok_start, tok_end) in zip(token_preds, offset_mapping):
        # Skip tokens with dummy offsets (e.g., special tokens) if desired.
        if pred == 1:
            if current_span is None:
                current_span = [tok_start, tok_end]
            else:
                # Extend the span to the end of this token.
                current_span[1] = tok_end
        else:
            if current_span is not None:
                spans.append(tuple(current_span))
                current_span = None
    # Append any remaining span.
    if current_span is not None:
        spans.append(tuple(current_span))

    return {
        "class_probs": class_probs,
        "class_preds": class_preds,
        "token_probs": token_probs,
        "token_preds": token_preds,
        "spans": spans
    }


In [ ]:
all_labels = set()
train['techniques'].apply(lambda x: all_labels.update(x))

In [ ]:
# Predicting for test:

predictions = []
for text in tqdm(test.content.values):
    predictions.append(run_inference(model, tokenizer, text, device=DEVICE, class_thresholds=class_thresholds, token_threshold=token_threshold))

  0%|          | 0/5735 [00:00<?, ?it/s]

In [ ]:
# Saving prediction for token:
pred_token = pd.DataFrame({
    "id": test.id.values
})

pred_values = [p["spans"] for p in predictions]
pred_token["trigger_words"] = pred_values

pred_token.to_csv("pred_span_xlm_roberta_large_multitask.csv", index=False)

## Calculate score

In [2]:
import ast
import pandas as pd

In [6]:
def span_f1(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Compute span-level F1 score based on overlap.

    Parameters:
    - solution (pd.DataFrame): Ground truth DataFrame with row ID and token labels.
    - submission (pd.DataFrame): Submission DataFrame with row ID and token labels.
    - row_id_column_name (str): Column name for the row identifier.

    Returns:
    - float: The token-level weighted F1 score.

    Example:
    >>> solution = pd.DataFrame({
    ...     "id": [1, 2, 3],
    ...     "trigger_words": [[(612, 622), (725, 831)], [(300, 312)], []]
    ... })
    >>> submission = pd.DataFrame({
    ...     "id": [1, 2, 3],
    ...     "trigger_words": [[(612, 622), (700, 720)], [(300, 312)], [(100, 200)]]
    ... })
    >>> score(solution, submission, "id")
    0.16296296296296295
    """
    if not all(col in solution.columns for col in ["id", "trigger_words"]):
        raise ValueError("Solution DataFrame must contain 'id' and 'trigger_words' columns.")
    if not all(col in submission.columns for col in ["id", "trigger_words"]):
        raise ValueError("Submission DataFrame must contain 'id' and 'trigger_words' columns.")
    
    def safe_parse_spans(trigger_words):
        if isinstance(trigger_words, str):
            try:
                return ast.literal_eval(trigger_words)
            except (ValueError, SyntaxError):
                return []
        if isinstance(trigger_words, (list, tuple)):
            return trigger_words
        return []

    def extract_tokens_from_spans(spans):
        tokens = set()
        for start, end in spans:
            tokens.update(range(start, end))
        return tokens
    
    solution = solution.copy()
    submission = submission.copy()

    solution["trigger_words"] = solution["trigger_words"].apply(safe_parse_spans)
    submission["trigger_words"] = submission["trigger_words"].apply(safe_parse_spans)

    merged = pd.merge(
        solution,
        submission,
        on="id",
        suffixes=("_solution", "_submission")
    )

    total_true_tokens = 0
    total_pred_tokens = 0
    overlapping_tokens = 0

    for _, row in merged.iterrows():
        true_spans = row["trigger_words_solution"]
        pred_spans = row["trigger_words_submission"]

        true_tokens = extract_tokens_from_spans(true_spans)
        pred_tokens = extract_tokens_from_spans(pred_spans)

        total_true_tokens += len(true_tokens)
        total_pred_tokens += len(pred_tokens)
        overlapping_tokens += len(true_tokens & pred_tokens)

    precision = overlapping_tokens / total_pred_tokens if total_pred_tokens > 0 else 0
    recall = overlapping_tokens / total_true_tokens if total_true_tokens > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return f1

In [7]:
best_span = pd.read_csv(os.path.join(TRAIN_PATH, "pred_span_xlm_roberta_large_multitask.csv"))
best_span['trigger_words'] = best_span['trigger_words'].apply(ast.literal_eval)

In [10]:
f1 = span_f1(best_span, test, row_id_column_name="id")
print("Span-level F1 score:", f1)

Span-level F1 score: 0.5988803556149175
